In [ ]:
import sys
import warnings

sys.path.append('../src')

warnings.filterwarnings('ignore', category=RuntimeWarning)

import pandas as pd
from models.clustering import (
    load_scaled_rfm,
    train_kmeans,
    train_gmm,
    train_hdbscan,
    evaluate_model,
    compare_models,
    save_cluster_result,
)

## 1. Đọc và kiểm tra dữ liệu đầu vào

Kiểm tra số lượng khách hàng, số đặc trưng, giá trị thiếu và kiểu dữ liệu.

In [2]:
df, X = load_scaled_rfm('../data/processed/rfm_scaled.csv')
df.head()

Loaded ../data/processed/rfm_scaled.csv
  Customers: 4324
  Features : ['Recency', 'Frequency', 'Monetary']
  Missing  : {'Recency': 0, 'Frequency': 0, 'Monetary': 0}
  Dtypes   : {'Recency': dtype('float64'), 'Frequency': dtype('float64'), 'Monetary': dtype('float64')}


,CustomerID,Recency,Frequency,Monetary
0,12347.0,-0.901281,1.088034,1.445284
1,12348.0,-0.173081,0.396756,0.567579
2,12349.0,-0.741675,-0.950919,0.578791
3,12350.0,2.171126,-0.950919,-0.697983
4,12352.0,-0.562119,0.891637,0.465850


In [3]:
df[['Recency', 'Frequency', 'Monetary']].describe()

,Recency,Frequency,Monetary
count,4.324000e+03,4.324000e+03,4.324000e+03
mean,4.683272e-17,9.613032e-17,-9.859520e-17
std,1.000116e+00,1.000116e+00,1.000116e+00
min,-9.212316e-01,-9.509185e-01,-4.157915e+00
25%,-7.516507e-01,-9.509185e-01,-6.797203e-01
50%,-4.224642e-01,-3.545631e-01,-5.833738e-02
75%,5.052430e-01,6.649133e-01,6.607762e-01
max,2.799573e+00,5.851461e+00,4.778701e+00


## 2. Huấn luyện K-Means

Huấn luyện với `n_clusters` từ 2 đến 10, ghi lại nhãn cụm, inertia và thời gian huấn luyện.

In [4]:
kmeans_rows = []
for k in range(2, 11):
    result = train_kmeans(X, k)
    kmeans_rows.append({
        'n_clusters': k,
        'inertia': result['inertia'],
        'train_time': result['train_time'],
        **evaluate_model(X, result['labels']),
    })

pd.DataFrame(kmeans_rows)

,n_clusters,inertia,train_time,silhouette,davies_bouldin,calinski_harabasz
0,2,6839.618587,0.017056,0.406297,0.914419,3875.187644
1,3,4278.752298,0.003795,0.414329,0.829263,4389.589669
2,4,3212.395105,0.003481,0.379564,0.857874,4375.080072
3,5,2735.154437,0.003285,0.344008,0.938325,4041.220488
4,6,2365.531204,0.005235,0.332324,0.972073,3872.351145
5,7,2141.929433,0.005011,0.330648,0.931656,3638.162706
6,8,1930.637526,0.006887,0.304955,0.989923,3526.193181
7,9,1826.832396,0.005518,0.296794,1.062505,3290.701083
8,10,1701.629363,0.007591,0.292932,1.058250,3174.759700


## 3. Huấn luyện GMM

Huấn luyện với `n_components` từ 2 đến 10 và bốn dạng `covariance_type`,
ghi lại nhãn cụm, log-likelihood và thời gian huấn luyện.

In [5]:
gmm_rows = []
for covariance_type in ('full', 'tied', 'diag', 'spherical'):
    for k in range(2, 11):
        result = train_gmm(X, k, covariance_type)
        gmm_rows.append({
            'n_components': k,
            'covariance_type': covariance_type,
            'log_likelihood': result['log_likelihood'],
            'train_time': result['train_time'],
            **evaluate_model(X, result['labels']),
        })

gmm_results = pd.DataFrame(gmm_rows)
gmm_results.loc[gmm_results.groupby('covariance_type')['silhouette'].idxmax()]

,n_components,covariance_type,log_likelihood,train_time,silhouette,davies_bouldin,calinski_harabasz
18,2,diag,-14821.894871,0.005820,0.392589,0.941722,3693.009042
0,2,full,-13064.306980,0.010435,0.303118,1.200069,2165.261750
28,3,spherical,-15259.944945,0.006251,0.414446,0.816665,4364.692454
10,3,tied,-13903.954176,0.011229,0.377866,0.824813,3487.493291


## 4. Huấn luyện HDBSCAN

Huấn luyện với các giá trị `min_cluster_size` và `min_samples`,
ghi lại số cụm phát hiện được, số điểm nhiễu và thời gian huấn luyện.

HDBSCAN được lấy từ `sklearn.cluster.HDBSCAN`.

In [6]:
hdbscan_rows = []
for min_cluster_size in (5, 10, 20, 30, 50):
    for min_samples in (None, 5, 10):
        result = train_hdbscan(X, min_cluster_size, min_samples)
        hdbscan_rows.append({
            'min_cluster_size': min_cluster_size,
            'min_samples': min_samples,
            'n_clusters': result['n_clusters'],
            'n_noise': result['n_noise'],
            'train_time': result['train_time'],
            # Loại điểm nhiễu trước khi tính chỉ số đánh giá
            **evaluate_model(X, result['labels'], drop_noise=True),
        })

pd.DataFrame(hdbscan_rows)

,min_cluster_size,min_samples,n_clusters,n_noise,train_time,silhouette,davies_bouldin,calinski_harabasz
0,5,NaN,32,522,0.047591,-0.038626,1.386552,174.005250
1,5,5.0,32,522,0.042479,-0.038626,1.386552,174.005250
2,5,10.0,9,694,0.043631,-0.028520,1.994623,640.214426
3,10,NaN,9,694,0.041511,-0.028520,1.994623,640.214426
4,10,5.0,15,518,0.041073,0.027180,2.099683,362.425770
5,10,10.0,9,694,0.041222,-0.028520,1.994623,640.214426
6,20,NaN,6,893,0.044125,0.057080,1.752768,970.233420
7,20,5.0,10,484,0.039269,0.024506,2.514048,577.608302
8,20,10.0,8,651,0.040110,0.036089,2.150857,718.279778
9,30,NaN,5,1033,0.046510,0.087352,1.537856,1239.064201


## 5. Tổng hợp và so sánh

`compare_models` chạy lại toàn bộ 60 cấu hình của ba thuật toán và gom kết quả vào một bảng.

In [7]:
results = compare_models(X)
print(f"Tổng số cấu hình: {len(results)}")

best_per_model = results.loc[results.groupby('model')['silhouette'].idxmax()]
best_per_model[['model', 'params', 'n_clusters', 'n_noise',
                'silhouette', 'davies_bouldin', 'calinski_harabasz', 'train_time']]

Tổng số cấu hình: 60


,model,params,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,train_time
37,GMM,"n_components=3, covariance_type=spherical",3,0,0.414446,0.816665,4364.692454,0.005991
57,HDBSCAN,"min_cluster_size=50, min_samples=None",3,854,0.202819,1.195553,2404.568174,0.052019
1,K-Means,n_clusters=3,3,0,0.414329,0.829263,4389.589669,0.003950


## 6. Lưu kết quả 

Mô hình được chọn là K-Means với 3 cụm. Căn cứ lựa chọn được trình bày tại
`outputs/reports/clustering_comparison.md` mục 6.4.

In [8]:
final_labels = train_kmeans(X, 3)['labels']
clusters = save_cluster_result(df['CustomerID'], final_labels, '../data/processed/customer_clusters.csv')
clusters['Cluster'].value_counts().sort_index()

Saved 4324 cluster labels to ../data/processed/customer_clusters.csv


Cluster
0     990
1    2024
2    1310
Name: count, dtype: int64

## 7. Đặc điểm ba cụm theo giá trị RFM gốc

Ghép nhãn cụm với bảng RFM chưa chuẩn hóa để kiểm tra ý nghĩa nghiệp vụ của từng cụm.

In [9]:
rfm_raw = pd.read_csv('../data/processed/rfm_table.csv')
profile = rfm_raw.merge(clusters, on='CustomerID')

profile.groupby('Cluster').agg(
    n_customers=('CustomerID', 'count'),
    recency_mean=('Recency', 'mean'),
    frequency_mean=('Frequency', 'mean'),
    monetary_mean=('Monetary', 'mean'),
    monetary_median=('Monetary', 'median'),
).round(1)

,n_customers,recency_mean,frequency_mean,monetary_mean,monetary_median
Cluster,,,,,
0,990,254.3,1.4,400.7,282.4
1,2024,54.0,2.0,601.3,501.4
2,1310,29.1,9.8,5114.4,2498.0
